In [2]:
import os, json, time, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, matthews_corrcoef, confusion_matrix)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

EPOCHS, BATCH_SIZE, TEST_SIZE = 40, 512, 0.30
LR, MOMENTUM = 0.01, 0.9
OUTPUT_DIR = "/kaggle/working"
RESULTS_PATH = os.path.join(OUTPUT_DIR, "stability_results.json")

hits = glob.glob("/kaggle/input/**/ciciot2023_working_set.parquet", recursive=True)
df = pd.read_parquet(hits[0])
feature_cols = [c for c in df.columns if c != "family"]
X_all = df[feature_cols].to_numpy(dtype=np.float32)
le = LabelEncoder(); y_all = le.fit_transform(df["family"].to_numpy())
CLASS_NAMES = list(le.classes_)
N_FEATURES, N_CLASSES = X_all.shape[1], len(CLASS_NAMES)
print(f"X: {X_all.shape} | {N_CLASSES} classes")

def build_model(n_inputs, n_output):
    nb = int(round(n_inputs / 2.0))
    visible = keras.Input(shape=(n_inputs, 1))
    e = layers.Dense(n_inputs)(visible); e = layers.BatchNormalization()(e); e = layers.LeakyReLU()(e)
    bottleneck = layers.Dense(nb)(e)
    d = layers.Dense(n_inputs)(bottleneck); d = layers.BatchNormalization()(d); d = layers.LeakyReLU()(d)
    lstm = layers.LSTM(nb, activation="tanh", return_sequences=True)(visible)
    lstm = layers.Dense(n_inputs)(lstm)
    c = layers.Concatenate()([d, lstm])
    c = layers.Conv1D(filters=nb, kernel_size=2, activation="relu")(c)
    c = layers.Flatten()(c)
    out = layers.Dense(n_output, activation="softmax")(c)
    m = keras.Model(visible, out)
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=LR, momentum=MOMENTUM),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m

def run_stability(protocol, seed):
    """Protocol B/A, strategy=none. No probability files saved."""
    t0 = time.time()
    np.random.seed(seed); tf.random.set_seed(seed)
    idx = np.arange(len(X_all))
    idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE,
                                      random_state=seed, stratify=y_all)
    if protocol == "A":
        sc = StandardScaler().fit(X_all); Xs = sc.transform(X_all)
        X_tr, y_tr = Xs[idx_tr], y_all[idx_tr]
        X_te, y_te = Xs[idx_te], y_all[idx_te]
    else:
        sc = StandardScaler().fit(X_all[idx_tr])
        X_tr, y_tr = sc.transform(X_all[idx_tr]), y_all[idx_tr]
        X_te, y_te = sc.transform(X_all[idx_te]), y_all[idx_te]

    y_tr_oh = keras.utils.to_categorical(y_tr, N_CLASSES)
    X_tr = X_tr.reshape(-1, N_FEATURES, 1).astype(np.float32)
    X_te = X_te.reshape(-1, N_FEATURES, 1).astype(np.float32)

    model = build_model(N_FEATURES, N_CLASSES)
    hist = model.fit(X_tr, y_tr_oh, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    y_pred = model.predict(X_te, batch_size=2048, verbose=0).argmax(axis=1)

    cm = confusion_matrix(y_te, y_pred)
    res = {
        "protocol": protocol, "seed": seed,
        "accuracy": float(accuracy_score(y_te, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_te, y_pred)),
        "macro_f1": float(f1_score(y_te, y_pred, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_te, y_pred)),
        "final_train_loss": float(hist.history["loss"][-1]),
        "min_train_loss": float(min(hist.history["loss"])),
        "loss_curve": [float(v) for v in hist.history["loss"]],
        "confusion_matrix": cm.tolist(),
        "n_empty_pred_classes": int((cm.sum(axis=0) == 0).sum()),
        "max_pred_share": float(cm.sum(axis=0).max() / cm.sum()),
        "wall_sec": round(time.time() - t0, 1),
    }
    keras.backend.clear_session()
    return res

print("Ready.")

X: (547944, 44) | 8 classes
Ready.


In [3]:
from sklearn.metrics import recall_score

def run_stability(protocol, seed):
    """Records train accuracy, per-epoch validation, and best-epoch rescue."""
    t0 = time.time()
    np.random.seed(seed); tf.random.set_seed(seed)

    idx = np.arange(len(X_all))
    idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE,
                                      random_state=seed, stratify=y_all)
    if protocol == "A":
        sc = StandardScaler().fit(X_all); Xs = sc.transform(X_all)
        X_tr, y_tr = Xs[idx_tr], y_all[idx_tr]
        X_te, y_te = Xs[idx_te], y_all[idx_te]
    else:
        sc = StandardScaler().fit(X_all[idx_tr])
        X_tr, y_tr = sc.transform(X_all[idx_tr]), y_all[idx_tr]
        X_te, y_te = sc.transform(X_all[idx_te]), y_all[idx_te]

    # carve a validation slice out of TRAIN only — test stays untouched
    i_fit, i_val = train_test_split(np.arange(len(y_tr)), test_size=0.10,
                                    random_state=seed, stratify=y_tr)
    X_fit, y_fit = X_tr[i_fit], y_tr[i_fit]
    X_val, y_val = X_tr[i_val], y_tr[i_val]

    rs = lambda a: a.reshape(-1, N_FEATURES, 1).astype(np.float32)
    X_fit, X_val, X_te_r = rs(X_fit), rs(X_val), rs(X_te)
    y_fit_oh = keras.utils.to_categorical(y_fit, N_CLASSES)
    y_val_oh = keras.utils.to_categorical(y_val, N_CLASSES)

    ckpt = f"/kaggle/working/_best_{protocol}_{seed}.weights.h5"
    model = build_model(N_FEATURES, N_CLASSES)
    hist = model.fit(
        X_fit, y_fit_oh, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
        validation_data=(X_val, y_val_oh),
        callbacks=[keras.callbacks.ModelCheckpoint(
            ckpt, monitor="val_accuracy", mode="max",
            save_best_only=True, save_weights_only=True, verbose=0)])

    def evaluate(m):
        p = m.predict(X_te_r, batch_size=2048, verbose=0).argmax(1)
        cm = confusion_matrix(y_te, p)
        return {
            "accuracy": float(accuracy_score(y_te, p)),
            "balanced_accuracy": float(balanced_accuracy_score(y_te, p)),
            "macro_f1": float(f1_score(y_te, p, average="macro", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_te, p)),
            "confusion_matrix": cm.tolist(),
            "n_empty_pred_classes": int((cm.sum(0) == 0).sum()),
            "max_pred_share": float(cm.sum(0).max() / cm.sum()),
        }

    final_eval = evaluate(model)                      # what you'd ship at epoch 40
    train_pred = model.predict(X_fit, batch_size=2048, verbose=0).argmax(1)
    train_acc = float(accuracy_score(y_fit, train_pred))
    train_bal = float(balanced_accuracy_score(y_fit, train_pred))

    model.load_weights(ckpt)                          # rescue by val-selected epoch
    rescued_eval = evaluate(model)
    os.remove(ckpt)

    va = hist.history["val_accuracy"]
    res = {
        "protocol": protocol, "seed": seed,
        **final_eval,
        "train_accuracy": train_acc,
        "train_balanced_accuracy": train_bal,
        "generalisation_gap": train_acc - final_eval["accuracy"],
        "rescued_accuracy": rescued_eval["accuracy"],
        "rescued_balanced_accuracy": rescued_eval["balanced_accuracy"],
        "rescued_mcc": rescued_eval["mcc"],
        "best_val_epoch": int(np.argmax(va)) + 1,
        "best_val_accuracy": float(max(va)),
        "final_val_accuracy": float(va[-1]),
        "val_acc_epoch5": float(va[4]) if len(va) > 4 else None,
        "final_train_loss": float(hist.history["loss"][-1]),
        "loss_curve": [float(v) for v in hist.history["loss"]],
        "val_loss_curve": [float(v) for v in hist.history["val_loss"]],
        "val_acc_curve": [float(v) for v in va],
        "wall_sec": round(time.time() - t0, 1),
    }
    keras.backend.clear_session()
    return res

print("run_stability updated.")

run_stability updated.


In [4]:
N_SEEDS = 20
PROTOCOLS_TO_RUN = ["B", "A"]

results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else []
done = {(r["protocol"], r["seed"]) for r in results}
grid = [(p, s) for p in PROTOCOLS_TO_RUN for s in range(N_SEEDS)]
print(f"{len(grid)} runs, {len(done)} already done\n" + "="*78)

for k, (proto, seed) in enumerate(grid, 1):
    if (proto, seed) in done:
        print(f"[{k}/{len(grid)}] skip {proto}/s{seed}"); continue
    print(f"[{k}/{len(grid)}] protocol={proto} seed={seed} ...", flush=True)
    try:
        r = run_stability(proto, seed)
        results.append(r)
        json.dump(results, open(RESULTS_PATH, "w"), indent=2)
        flag = "  <-- COLLAPSE" if r["accuracy"] < 0.60 else ""
        saved = "  [RESCUED]" if (r["accuracy"] < 0.60 and r["rescued_accuracy"] > 0.60) else ""
        print(f"    test={r['accuracy']:.4f}  train={r['train_accuracy']:.4f}  "
              f"gap={r['generalisation_gap']:+.4f}  loss={r['final_train_loss']:.4f}")
        print(f"    rescued={r['rescued_accuracy']:.4f} (ep {r['best_val_epoch']})  "
              f"val@5={r['val_acc_epoch5']:.4f}  empty={r['n_empty_pred_classes']}"
              f"  ({r['wall_sec']}s){flag}{saved}")
    except Exception as e:
        import traceback; print(f"    FAILED: {e}"); traceback.print_exc()

print("\nDone ->", RESULTS_PATH)

40 runs, 0 already done
[1/40] protocol=B seed=0 ...
    test=0.8484  train=0.8498  gap=+0.0014  loss=0.4285
    rescued=0.8764 (ep 27)  val@5=0.8072  empty=0  (242.8s)
[2/40] protocol=B seed=1 ...
    test=0.8034  train=0.8050  gap=+0.0015  loss=0.3837
    rescued=0.8484 (ep 20)  val@5=0.8121  empty=0  (237.6s)
[3/40] protocol=B seed=2 ...
    test=0.7939  train=0.7947  gap=+0.0008  loss=0.3376
    rescued=0.8690 (ep 26)  val@5=0.8037  empty=0  (238.7s)
[4/40] protocol=B seed=3 ...
    test=0.8308  train=0.8314  gap=+0.0006  loss=0.3776
    rescued=0.8618 (ep 28)  val@5=0.8042  empty=0  (238.3s)
[5/40] protocol=B seed=4 ...
    test=0.8068  train=0.8097  gap=+0.0029  loss=0.3465
    rescued=0.8823 (ep 34)  val@5=0.8012  empty=0  (236.7s)
[6/40] protocol=B seed=5 ...
    test=0.8271  train=0.8302  gap=+0.0031  loss=0.4104
    rescued=0.8763 (ep 26)  val@5=0.8022  empty=0  (239.7s)
[7/40] protocol=B seed=6 ...
    test=0.8150  train=0.8136  gap=-0.0014  loss=0.3972
    rescued=0.8721 (e

In [1]:
from IPython.display import FileLink
import os
p = "/kaggle/working/stability_results.json"
print("exists:", os.path.exists(p), "| size:", os.path.getsize(p), "bytes")
FileLink(p)

exists: True | size: 197209 bytes


/kaggle/working/stability_results.json

In [2]:
import os, glob, json
p = "/kaggle/working/stability_results.json"
print("exists:", os.path.exists(p))
if os.path.exists(p):
    print("size:", os.path.getsize(p), "bytes")
    r = json.load(open(p))
    print("records:", len(r))
    have = {(x["protocol"], x["seed"]) for x in r}
    for proto in ["A", "B"]:
        print(f"  protocol {proto}: {sum(1 for k in have if k[0]==proto)}/20")
print("\nAll files in /kaggle/working:")
for f in sorted(os.listdir("/kaggle/working")):
    print(f"  {f}  ({os.path.getsize('/kaggle/working/'+f)} bytes)")

exists: True
size: 197209 bytes
records: 40
  protocol A: 20/20
  protocol B: 20/20

All files in /kaggle/working:
  .virtual_documents  (4096 bytes)
  stability_results.json  (197209 bytes)
